# Modelo XGBoost con metodo por Transecto y metodo General

In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
XGBoost con MultiOutputRegressor y búsqueda de hiperparámetros.
"""

import os
import json
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

BASE_DIR = os.path.expanduser("/Volumes/copia seguridad1/enviar_benja/carpeta sin título/clean/Finales/")
WINDOWS_PARTITIONED_DIR = os.path.join(BASE_DIR, "windows_partitioned")
MODELS_DIR = os.path.join(BASE_DIR, "models")
ENCODED_DIR = os.path.join(BASE_DIR, "encoded")  # para cargar CSV originales (necesario para estaciones)

DATA_DIR = WINDOWS_PARTITIONED_DIR
OUTPUT_DIR = os.path.join(MODELS_DIR, "xgboost")
os.makedirs(OUTPUT_DIR, exist_ok=True)

WINDOW_IN = 72
WINDOW_OUT = 72
RANDOM_STATE = 42
N_JOBS = -1

XGB_PARAM_GRID = {
    'estimator__n_estimators': [200],
    'estimator__max_depth': [8],
    'estimator__learning_rate': [0.1],
    'estimator__subsample': [1],
    'estimator__colsample_bytree': [1],
    'estimator__reg_alpha': [0.1],
    'estimator__reg_lambda': [1]
}


def willmott_index(y_true, y_pred):
    numer = np.sum((y_true - y_pred) ** 2)
    denom = np.sum((np.abs(y_pred - y_true.mean()) + np.abs(y_true - y_true.mean())) ** 2)
    return 1 - numer / denom if denom != 0 else np.nan


def mape(y_true, y_pred):
    mask = y_true != 0
    if not mask.any():
        return np.nan
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100


def compute_metrics(y_true, y_pred):
    return {
        'r2': r2_score(y_true, y_pred),
        'mae': mean_absolute_error(y_true, y_pred),
        'rmse': np.sqrt(mean_squared_error(y_true, y_pred)),
        'mape': mape(y_true, y_pred),
        'willmott': willmott_index(y_true, y_pred)
    }


def plot_predictions(y_true, y_pred, horizons, save_path, title):
    n_plots = len(horizons)
    fig, axes = plt.subplots(1, n_plots, figsize=(5*n_plots, 4))
    if n_plots == 1:
        axes = [axes]
    for ax, h in zip(axes, horizons):
        ax.scatter(y_true[:, h], y_pred[:, h], alpha=0.3, s=10)
        ax.plot([y_true.min(), y_true.max()], [y_true.min(), y_true.max()], 'r--', lw=1)
        ax.set_xlabel('Real O3 (µg/m³)')
        ax.set_ylabel('Predicho O3 (µg/m³)')
        ax.set_title(f'Horizonte {h+1}h')
        ax.grid(True, alpha=0.3)
    plt.suptitle(title)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()


def get_station_from_features(X_sample, feature_names, station_prefix='Estacion_'):
    n_features = len(feature_names)
    first_block = X_sample[:n_features]
    station_indices = [i for i, name in enumerate(feature_names) if name.startswith(station_prefix)]
    for idx in station_indices:
        if abs(first_block[idx] - 1.0) < 0.1:
            return feature_names[idx][len(station_prefix):]
    return None


def compute_per_station_metrics(y_true, y_pred, X_test, feature_names, output_dir, entity_name):
    n_samples = len(y_true)
    station_preds = {}
    for i in range(n_samples):
        station = get_station_from_features(X_test[i], feature_names)
        if station is None:
            continue
        station_preds.setdefault(station, {'true': [], 'pred': []})
        station_preds[station]['true'].append(y_true[i])
        station_preds[station]['pred'].append(y_pred[i])
    station_metrics = {}
    for station, data in station_preds.items():
        true_stack = np.vstack(data['true'])
        pred_stack = np.vstack(data['pred'])
        metrics = compute_metrics(true_stack.ravel(), pred_stack.ravel())
        station_metrics[station] = metrics
    if station_metrics:
        df = pd.DataFrame(station_metrics).T
        df.index.name = 'station'
        df.to_csv(os.path.join(output_dir, f"{entity_name}_per_station_metrics.csv"))
    return station_metrics


def train_and_evaluate_xgb(X_train, y_train, X_val, X_test, y_val, y_test,
                           entity_name, output_subdir, feature_names=None, original_csv_path=None):
    print(f"\n--- Entrenando XGBoost para {entity_name} ---")
    if len(X_val) == 0 or len(y_val) == 0:
        print(f"  Saltando {entity_name}: sin datos de validación.")
        return None, None

    base_model = xgb.XGBRegressor(objective='reg:squarederror', random_state=RANDOM_STATE)
    multi_model = MultiOutputRegressor(base_model, n_jobs=1)
    tscv = TimeSeriesSplit(n_splits=3)
    grid_search = GridSearchCV(estimator=multi_model, param_grid=XGB_PARAM_GRID, cv=tscv,
                               scoring='neg_mean_squared_error', n_jobs=N_JOBS, verbose=2)
    print("  Buscando mejores hiperparámetros...")
    grid_search.fit(X_train, y_train)
    best_model = grid_search.best_estimator_
    best_params = {k.replace('estimator__', ''): v for k, v in grid_search.best_params_.items()}
    print(f"  Mejores parámetros: {best_params}")

    val_pred = best_model.predict(X_val)
    val_metrics = compute_metrics(y_val.ravel(), val_pred.ravel())
    print(f"  Métricas en validación: R2={val_metrics['r2']:.3f}, MAE={val_metrics['mae']:.2f}")

    test_pred = best_model.predict(X_test)
    test_metrics = compute_metrics(y_test.ravel(), test_pred.ravel())
    print(f"  Métricas en test: R2={test_metrics['r2']:.3f}, MAE={test_metrics['mae']:.2f}, RMSE={test_metrics['rmse']:.2f}")

    if feature_names is not None and original_csv_path is not None:
        station_metrics = compute_per_station_metrics(y_test, test_pred, X_test, feature_names, output_subdir, entity_name)
    else:
        station_metrics = None

    with open(os.path.join(output_subdir, "model.pkl"), 'wb') as f:
        pickle.dump(best_model, f)
    results = {'best_params': best_params, 'validation_metrics': val_metrics, 'test_metrics': test_metrics,
               'station_metrics': station_metrics, 'n_train': len(X_train), 'n_val': len(X_val), 'n_test': len(X_test)}
    with open(os.path.join(output_subdir, "results.json"), 'w') as f:
        json.dump(results, f, indent=2)

    horizons = [23, 47, 71]
    plot_predictions(y_test, test_pred, horizons, os.path.join(output_subdir, "test_scatter.png"),
                     f"XGBoost - {entity_name} - Test")
    np.save(os.path.join(output_subdir, "test_pred.npy"), test_pred)
    np.save(os.path.join(output_subdir, "test_true.npy"), y_test)
    return best_model, test_metrics


def process_by_transect():
    print("\n" + "="*50)
    print("PROCESANDO XGBOOST POR TRANSECTO")
    ml_2d_dir = os.path.join(DATA_DIR, "by_transect", "ml")
    if not os.path.exists(ml_2d_dir):
        return
    entities = [d for d in os.listdir(ml_2d_dir) if os.path.isdir(os.path.join(ml_2d_dir, d))]
    for entity in entities:
        ml_2d_path = os.path.join(ml_2d_dir, entity, "ml_2d")
        if not os.path.exists(ml_2d_path):
            continue
        X_train = np.load(os.path.join(ml_2d_path, "train_X.npy"))
        y_train = np.load(os.path.join(ml_2d_path, "train_y.npy"))
        X_val   = np.load(os.path.join(ml_2d_path, "val_X.npy"))
        y_val   = np.load(os.path.join(ml_2d_path, "val_y.npy"))
        X_test  = np.load(os.path.join(ml_2d_path, "test_X.npy"))
        y_test  = np.load(os.path.join(ml_2d_path, "test_y.npy"))

        original_csv = os.path.join(ENCODED_DIR, "ml", "by_transect", f"{entity}.csv")
        feature_names = None
        if os.path.exists(original_csv):
            df_cols = pd.read_csv(original_csv, nrows=0, index_col=0)
            feature_names = df_cols.columns.tolist()

        out_subdir = os.path.join(OUTPUT_DIR, "by_transect", entity)
        os.makedirs(out_subdir, exist_ok=True)
        train_and_evaluate_xgb(X_train, y_train, X_val, X_test, y_val, y_test,
                               entity, out_subdir, feature_names, original_csv if os.path.exists(original_csv) else None)


def process_global():
    print("\n" + "="*50)
    print("PROCESANDO XGBOOST GLOBAL")
    ml_2d_dir = os.path.join(DATA_DIR, "global", "ml")
    if not os.path.exists(ml_2d_dir):
        return
    entities = [d for d in os.listdir(ml_2d_dir) if os.path.isdir(os.path.join(ml_2d_dir, d))]
    for entity in entities:
        ml_2d_path = os.path.join(ml_2d_dir, entity, "ml_2d")
        if not os.path.exists(ml_2d_path):
            continue
        X_train = np.load(os.path.join(ml_2d_path, "train_X.npy"))
        y_train = np.load(os.path.join(ml_2d_path, "train_y.npy"))
        X_val   = np.load(os.path.join(ml_2d_path, "val_X.npy"))
        y_val   = np.load(os.path.join(ml_2d_path, "val_y.npy"))
        X_test  = np.load(os.path.join(ml_2d_path, "test_X.npy"))
        y_test  = np.load(os.path.join(ml_2d_path, "test_y.npy"))

        out_subdir = os.path.join(OUTPUT_DIR, "global", entity)
        os.makedirs(out_subdir, exist_ok=True)
        train_and_evaluate_xgb(X_train, y_train, X_val, X_test, y_val, y_test,
                               entity, out_subdir, feature_names=None, original_csv_path=None)


def generate_summary():
    summary = []
    for t in ["by_transect", "global"]:
        dir_path = os.path.join(OUTPUT_DIR, t)
        if not os.path.exists(dir_path):
            continue
        for entity in os.listdir(dir_path):
            res_file = os.path.join(dir_path, entity, "results.json")
            if os.path.exists(res_file):
                with open(res_file, 'r') as f:
                    data = json.load(f)
                summary.append({'entity': entity, 'type': t, **data['test_metrics']})
    if summary:
        pd.DataFrame(summary).to_csv(os.path.join(OUTPUT_DIR, "summary_metrics.csv"), index=False)
        print("\nResumen guardado.")


if __name__ == "__main__":
    print("XGBOOST")
    #process_by_transect()
    process_global()
    generate_summary()

XGBOOST

PROCESANDO XGBOOST GLOBAL

--- Entrenando XGBoost para T1_E1_Alicante ---
  Buscando mejores hiperparámetros...
Fitting 3 folds for each of 1 candidates, totalling 3 fits
[CV] END estimator__colsample_bytree=1, estimator__learning_rate=0.1, estimator__max_depth=8, estimator__n_estimators=200, estimator__reg_alpha=0.1, estimator__reg_lambda=1, estimator__subsample=1; total time=124.8min
[CV] END estimator__colsample_bytree=1, estimator__learning_rate=0.1, estimator__max_depth=8, estimator__n_estimators=200, estimator__reg_alpha=0.1, estimator__reg_lambda=1, estimator__subsample=1; total time=154.0min
[CV] END estimator__colsample_bytree=1, estimator__learning_rate=0.1, estimator__max_depth=8, estimator__n_estimators=200, estimator__reg_alpha=0.1, estimator__reg_lambda=1, estimator__subsample=1; total time=166.4min
  Mejores parámetros: {'colsample_bytree': 1, 'learning_rate': 0.1, 'max_depth': 8, 'n_estimators': 200, 'reg_alpha': 0.1, 'reg_lambda': 1, 'subsample': 1}
  Métricas

# Grafico serie temporal O3 + Predicciones

In [4]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Gráficas de predicción de O₃ con XGBoost para múltiples estaciones.
Para cada estación:
- Muestra los últimos 7 días observados de O₃.
- Predice los siguientes 3 días (72h) usando el modelo entrenado.
Las gráficas se guardan en una carpeta 'xgboost_forecasts'.
"""

import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# =============================================================================
# CONFIGURACIÓN (AJUSTAR SEGÚN TU ESTRUCTURA DE DIRECTORIOS)
# =============================================================================
# Lista de estaciones a procesar
STATIONS = [
    "T1_E1_Alicante",
    "T1_E2_Elda",
    "T2_E1_Elche",
    "T2_E2_Elda",
    "T3_E1_Valencia",
    "T3_E2_Buñol",
    "T4_E1_Valencia",
    "T4_E2_Villar_Arzobispo",
    "T5_E1_Castellon",
    "T5_E2_Onda",
    "T6_E1_Sant_Jordi",
    "T6_E2_Coratxa",
    "T6_E3_Zorita",
    "T8_E1_Sant_Jordi",
    "T8_E2_Morella",
    "T8_E3_Zorita"
]

# Rutas base (cambiar según tu sistema)
BASE_DIR = os.path.expanduser("/Volumes/copia seguridad1/enviar_benja/carpeta sin título/clean/Finales/")
MODELS_BASE = os.path.join(BASE_DIR, "models", "xgboost", "global")   # dentro de cada estación hay model.pkl
ENCODED_DIR = os.path.join(BASE_DIR, "encoded", "ml", "global")       # archivos CSV con datos originales
OUTPUT_DIR = os.path.join(BASE_DIR, "xgboost_forecasts")              # carpeta donde guardar las gráficas

# Parámetros de ventana (deben coincidir con el entrenamiento)
WINDOW_IN = 72      # horas de entrada
WINDOW_OUT = 72     # horas de salida (predicción)

# =============================================================================
# FUNCIONES
# =============================================================================
def load_original_data(station):
    """
    Carga el CSV original con todas las variables horarias para una estación.
    El CSV debe tener un índice datetime (columna 'timestamp' o índice temporal).
    Retorna un DataFrame con frecuencias horarias completas.
    """
    csv_path = os.path.join(ENCODED_DIR, f"{station}.csv")
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"No se encontró el archivo: {csv_path}")
    df = pd.read_csv(csv_path, index_col=0, parse_dates=True)
    # Asegurar frecuencia horaria (rellenar huecos con NaN, opcionalmente podrías interpolar)
    df = df.asfreq('H')
    return df

def get_last_window(df, window_size=WINDOW_IN):
    """
    Extrae las últimas `window_size` filas del DataFrame (todas las variables).
    Si hay menos filas, lanza error.
    Retorna un array 2D de forma (window_size, n_features).
    """
    if len(df) < window_size:
        raise ValueError(f"Solo hay {len(df)} registros, se necesitan {window_size}")
    window = df.iloc[-window_size:].values  # (window_size, n_features)
    return window

def flatten_window(window):
    """Aplana la ventana 2D a un vector 1D de longitud window_size * n_features."""
    return window.reshape(1, -1)

def predict_next_72h(model, flat_window):
    """
    Usa el modelo XGBoost (MultiOutputRegressor) para predecir los siguientes 72 valores.
    flat_window: array de forma (1, n_features_total)
    Retorna array de predicciones de longitud WINDOW_OUT.
    """
    pred = model.predict(flat_window)  # forma (1, WINDOW_OUT)
    if pred.ndim == 2:
        pred = pred.flatten()
    return pred

def plot_and_save_forecast(station, historical_o3, forecast, forecast_start_date, output_dir):
    """
    Genera la gráfica y la guarda en output_dir.
    historical_o3: Serie de pandas con índice datetime (últimos 7 días de O3)
    forecast: array de 72 valores predichos
    forecast_start_date: datetime del primer valor predicho
    """
    forecast_index = pd.date_range(start=forecast_start_date, periods=len(forecast), freq='H')
    
    plt.figure(figsize=(14, 5))
    plt.plot(historical_o3.index, historical_o3.values, 'b-', linewidth=1.5, label='Observado (última semana)')
    plt.plot(forecast_index, forecast, 'r--', linewidth=2, label='Predicción XGBoost (72h)')
    plt.axvline(x=historical_o3.index[-1], color='gray', linestyle=':', label='Inicio predicción')
    plt.title(f"Predicción de O₃ con XGBoost - {station}\nÚltima semana observada + 3 días pronosticados")
    plt.xlabel("Fecha y hora")
    plt.ylabel("Concentración de O₃")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    
    # Guardar la figura
    os.makedirs(output_dir, exist_ok=True)
    out_path = os.path.join(output_dir, f"{station}_forecast.png")
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Gráfica guardada: {out_path}")

# =============================================================================
# PROGRAMA PRINCIPAL
# =============================================================================
if __name__ == "__main__":
    print("Generando predicciones para múltiples estaciones con XGBoost...")
    print(f"Directorio de salida: {OUTPUT_DIR}")
    
    for station in STATIONS:
        print(f"\n--- Procesando {station} ---")
        
        # 1. Ruta del modelo
        model_path = os.path.join(MODELS_BASE, station, "model.pkl")
        if not os.path.exists(model_path):
            print(f"  Modelo no encontrado en {model_path}. Se omite.")
            continue
        
        # 2. Cargar modelo
        try:
            with open(model_path, 'rb') as f:
                model = pickle.load(f)
            print(f"  Modelo cargado correctamente.")
        except Exception as e:
            print(f"  Error al cargar el modelo: {e}")
            continue
        
        # 3. Cargar datos originales de la estación
        try:
            df = load_original_data(station)
            print(f"  Datos cargados: {len(df)} registros, {df.shape[1]} variables.")
        except Exception as e:
            print(f"  Error al cargar datos: {e}")
            continue
        
        # 4. Verificar columna 'O3'
        if 'O3' not in df.columns:
            print(f"  El CSV no contiene columna 'O3'. Se omite.")
            continue
        
        # 5. Extraer última ventana de entrada (72h de todas las variables)
        try:
            last_window = get_last_window(df, WINDOW_IN)
            print(f"  Ventana de entrada extraída con forma {last_window.shape}")
        except Exception as e:
            print(f"  Error al extraer ventana: {e}")
            continue
        
        # 6. Aplanar ventana
        flat_input = flatten_window(last_window)
        print(f"  Entrada aplanada: {flat_input.shape}")
        
        # 7. Realizar predicción
        try:
            forecast = predict_next_72h(model, flat_input)
            print(f"  Predicción generada: {len(forecast)} valores")
        except Exception as e:
            print(f"  Error en predicción: {e}")
            continue
        
        # 8. Fecha de inicio del pronóstico (hora siguiente a la última observada)
        last_observed_time = df.index[-1] + pd.Timedelta(hours=1)
        
        # 9. Obtener últimos 7 días de O3 observado (168 horas)
        last_week_o3 = df['O3'].iloc[-168:]
        
        # 10. Generar y guardar gráfica
        try:
            plot_and_save_forecast(station, last_week_o3, forecast, last_observed_time, OUTPUT_DIR)
        except Exception as e:
            print(f"  Error al generar gráfica: {e}")
    
    print("\nProceso completado.")

Generando predicciones para múltiples estaciones con XGBoost...
Directorio de salida: /Volumes/copia seguridad1/enviar_benja/carpeta sin título/clean/Finales/xgboost_forecasts

--- Procesando T1_E1_Alicante ---


/Users/benjamincarbonell/Desktop/TFG/.conda/lib/python3.11/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator MultiOutputRegressor from version 1.8.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


  Modelo cargado correctamente.


/var/folders/85/2m2sr7rx6g5d561zn04dcq000000gn/T/ipykernel_25028/378203883.py:65: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df = df.asfreq('H')
/var/folders/85/2m2sr7rx6g5d561zn04dcq000000gn/T/ipykernel_25028/378203883.py:101: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  forecast_index = pd.date_range(start=forecast_start_date, periods=len(forecast), freq='H')


  Datos cargados: 140256 registros, 23 variables.
  Ventana de entrada extraída con forma (72, 23)
  Entrada aplanada: (1, 1656)
  Predicción generada: 72 valores
Gráfica guardada: /Volumes/copia seguridad1/enviar_benja/carpeta sin título/clean/Finales/xgboost_forecasts/T1_E1_Alicante_forecast.png

--- Procesando T1_E2_Elda ---


...

  Datos cargados: 149016 registros, 23 variables.
  Ventana de entrada extraída con forma (72, 23)
  Entrada aplanada: (1, 1656)
  Predicción generada: 72 valores
Gráfica guardada: /Volumes/copia seguridad1/enviar_benja/carpeta sin título/clean/Finales/xgboost_forecasts/T8_E2_Morella_forecast.png

--- Procesando T8_E3_Zorita ---


/Users/benjamincarbonell/Desktop/TFG/.conda/lib/python3.11/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator MultiOutputRegressor from version 1.8.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


  Modelo cargado correctamente.


/var/folders/85/2m2sr7rx6g5d561zn04dcq000000gn/T/ipykernel_25028/378203883.py:65: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df = df.asfreq('H')
/var/folders/85/2m2sr7rx6g5d561zn04dcq000000gn/T/ipykernel_25028/378203883.py:101: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  forecast_index = pd.date_range(start=forecast_start_date, periods=len(forecast), freq='H')


  Datos cargados: 149014 registros, 23 variables.
  Ventana de entrada extraída con forma (72, 23)
  Entrada aplanada: (1, 1656)
  Predicción generada: 72 valores
Gráfica guardada: /Volumes/copia seguridad1/enviar_benja/carpeta sin título/clean/Finales/xgboost_forecasts/T8_E3_Zorita_forecast.png

Proceso completado.
